In [1]:
import pandas as pd
import numpy as np

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
data = pd.read_csv(url)
data.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
print(data.shape)
print(data.info())
print(data.describe())
print(data.dtypes)

(10000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  object 
 2   fuel_type            10000 non-null  object 
 3   drivetrain           10000 non-null  object 
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB
None
         model_year    num_doors  engine_displacement  num_cylinders  \
count  10000.000000  10000.00000         10000.000000   10000.000000   
mean    1999.5495

## Question 1
### There's one column with missing values. What is it?

In [4]:
# We only need these specific columns for our model
columns_to_keep = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = data[columns_to_keep]
df.head(10)

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0
5,2290,266.0,4260,2007,31.2
6,2270,250.0,4310,1992,26.8
7,2370,279.0,4090,2005,30.2
8,2330,251.0,4230,1997,30.9
9,2330,280.0,4620,2004,31.2


In [5]:
missing_values = df.isnull().sum()
print(missing_values)

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


## Question 2
### What's the median (50% percentile) for variable 'horsepower'?

In [6]:
median_hp = df['horsepower'].median()
print(median_hp)

254.0


## Question 3
We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
* Which option gives better RMSE?

In [7]:
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [8]:
#filling missing horsepower with 0:

df_train = df_train.copy()
df_val = df_val.copy()

df_train["horsepower"] = df_train["horsepower"].fillna(0)
df_val["horsepower"] = df_val["horsepower"].fillna(0)
X_train = df_train[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_train = df_train.fuel_efficiency_mpg.values
X_val = df_val[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_val = df_val.fuel_efficiency_mpg.values

In [9]:
#Train the linear regression:

X_train = np.column_stack([np.ones(len(X_train)), X_train])
w = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train
y_pred = np.column_stack([np.ones(len(X_val)), X_val]) @ w
rmse = np.sqrt(((y_pred - y_val) ** 2).mean())
print(round(rmse, 3))

2.205


In [10]:
#Now doing the same thing using the mean:

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()

mean = df_train.horsepower.mean()

df_train["horsepower"] = df_train["horsepower"].fillna(mean)
df_val["horsepower"] = df_val["horsepower"].fillna(mean)
X_train = df_train[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_train = df_train.fuel_efficiency_mpg.values
X_val = df_val[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_val = df_val.fuel_efficiency_mpg.values
X_train = np.column_stack([np.ones(len(X_train)), X_train])
w = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train
y_pred = np.column_stack([np.ones(len(X_val)),X_val]) @ w
rmse = np.sqrt(((y_pred - y_val) ** 2).mean())

print(round(rmse, 3))

2.202


## Question 4
* Now let's train a regularized linear regression.
* For this question, fill the NAs with 0.
* Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
* Use RMSE to evaluate the model on the validation dataset.
* Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
* Which r gives the best RMSE?

In [11]:
# Now fill missing values with 0 and try each r.

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()

df_train["horsepower"] = df_train["horsepower"].fillna(0)
df_val["horsepower"] = df_val["horsepower"].fillna(0)

X_train = df_train[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_train = df_train.fuel_efficiency_mpg.values
X_val = df_val[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_val = df_val.fuel_efficiency_mpg.values

X_train = np.column_stack([np.ones(len(X_train)), X_train])
X_val = np.column_stack([np.ones(len(X_val)), X_val])

In [12]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    XTX = X_train.T @ X_train
    
    # Regularization
    XTX = XTX + r * np.eye(X_train.shape[1])
    w = np.linalg.inv(XTX) @ X_train.T @ y_train
    y_pred = X_val @ w
    rmse = np.sqrt(((y_pred - y_val) ** 2).mean())
    print(r, round(rmse, 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


## Question 5
We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
* Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
* For each seed, do the train/validation/test split with 60%/20%/20% distribution.
* Fill the missing values with 0 and train a model without regularization.
* For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
### What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
Round the result to 3 decimal digits (round(std, 3))
What's the value of std?

In [13]:
scores = []

for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy()
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()

    # Fill missing horsepower with 0
    df_train["horsepower"] = df_train["horsepower"].fillna(0)
    df_val["horsepower"] = df_val["horsepower"].fillna(0)
    X_train = df_train[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
    y_train = df_train.fuel_efficiency_mpg.values
    X_val = df_val[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
    y_val = df_val.fuel_efficiency_mpg.values

    # Add column of ones
    X_train = np.column_stack([np.ones(len(X_train)), X_train])
    X_val = np.column_stack([np.ones(len(X_val)), X_val])

    # Train model
    w = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train

    # Predict
    y_pred = X_val @ w

    # RMSE
    rmse = np.sqrt(((y_pred - y_val) ** 2).mean())

    scores.append(rmse)

print("Standard deviation:", round(np.std(scores), 3))

0.029


## Question 6
Split the dataset like previously, use seed 9.
- Combine train and validation datasets.
- Fill the missing values with 0 and train a model with r=0.001.
- What's the RMSE on the test dataset?

In [14]:
# Now let use seed 9, combine train and validation, and use r = 0.001.

np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

# Combine train and validation
df_train = pd.concat([df_train, df_val])

# Fill missing values
df_train = df_train.copy()
df_test = df_test.copy()

df_train["horsepower"] = df_train["horsepower"].fillna(0)
df_test["horsepower"] = df_test["horsepower"].fillna(0)

In [15]:
#Preparing the data:

X_train = df_train[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_train = df_train.fuel_efficiency_mpg.values
X_test = df_test[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]].values
y_test = df_test.fuel_efficiency_mpg.values
X_train = np.column_stack([np.ones(len(X_train)), X_train])
X_test = np.column_stack([np.ones(len(X_test)), X_test])

In [16]:
#Training with r = 0.001:

r = 0.001
XTX = X_train.T @ X_train
XTX = XTX + r * np.eye(X_train.shape[1])
w = np.linalg.inv(XTX) @ X_train.T @ y_train
y_pred = X_test @ w
rmse = np.sqrt(((y_pred - y_test) ** 2).mean())

print(round(rmse, 3))

2.236


# Let's compare with the LInear Regression Model

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

In [18]:
# Prepare the train/validation/test split
n = len(df)

val_n = int(n * 0.2)
test_n = int(n * 0.2)
train_n = n - val_n - test_n

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

dftrain = df.iloc[idx[:train_n]].copy()
dfval = df.iloc[idx[train_n:train_n + val_n]].copy()
dftest = df.iloc[idx[train_n + val_n:]].copy()

## Q3 — Fill missing values

We have two choices:

1. Fill missing horsepower with 0
2. Fill missing horsepower with the mean

We will test both.

In [19]:
#Fill missing horsepower with 0
dftrain["horsepower"] = dftrain["horsepower"].fillna(0)
dfval["horsepower"] = dfval["horsepower"].fillna(0)

In [20]:
#Select our X and targents

X_train = dftrain[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
X_val = dfval[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]

y_train = dftrain["fuel_efficiency_mpg"]
y_val = dfval["fuel_efficiency_mpg"]

In [21]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [22]:
# Make Prediction
y_pred = model.predict(X_val)

In [23]:
# Lets calculate RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(round(rmse, 3))

2.205


In [24]:
# Fill missing horsepower with the mean
dftrain = df.iloc[idx[:train_n]].copy()
dfval = df.iloc[idx[train_n:train_n + val_n]].copy()

In [25]:
mean = dftrain["horsepower"].mean()
print("Mean horsepower:", mean)

Mean horsepower: 254.46338337605272


In [26]:
dftrain["horsepower"] = dftrain["horsepower"].fillna(mean)
dfval["horsepower"] = dfval["horsepower"].fillna(mean)

In [27]:
X_train = dftrain[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
X_val = dfval[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]

y_train = dftrain["fuel_efficiency_mpg"]
y_val = dfval["fuel_efficiency_mpg"]

In [28]:
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [29]:
y_pred = model.predict(X_val)

In [30]:
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print( round(rmse, 3))

2.202


## Q4 — Regularization
We'll use Ridge

In [31]:
dftrain = df.iloc[idx[:train_n]].copy()
dfval = df.iloc[idx[train_n:train_n + val_n]].copy()

In [32]:
dftrain["horsepower"] = dftrain["horsepower"].fillna(0)
dfval["horsepower"] = dfval["horsepower"].fillna(0)

In [33]:
X_train = dftrain[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
X_val = dfval[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]

y_train = dftrain["fuel_efficiency_mpg"]
y_val = dfval["fuel_efficiency_mpg"]

In [34]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:

    model = Ridge(alpha=r)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(r, round(rmse, 4))

0 2.2053
0.01 2.2053
0.1 2.2053
1 2.2053
5 2.2053
10 2.2053
100 2.2053


## Q5  Different random seeds

In [40]:
scores = []

for seed in range(10):

    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    dftrain = df.iloc[idx[:train_n]].copy()

    df_val = df.iloc[idx[train_n:train_n + val_n]].copy()

    # Fill missing horsepower with 0
    dftrain["horsepower"] = dftrain["horsepower"].fillna(0)
    dfval["horsepower"] = dfval["horsepower"].fillna(0)

    # Select features
    X_train = dftrain[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
    X_val = dfval[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]

    # Select target
    y_train = dftrain["fuel_efficiency_mpg"]
    y_val = dfval["fuel_efficiency_mpg"]

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_val)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    scores.append(rmse)

std = np.std(scores)
print("Standard deviation:", round(std, 3))

Standard deviation: 0.001


## Q6

In [44]:
np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)

dftrain = df.iloc[idx[:train_n]].copy()
dfval = df.iloc[idx[train_n:train_n + val_n]].copy()
dftest = df.iloc[idx[train_n + val_n:]].copy()

dftrain = pd.concat([dftrain, dfval])

dftrain["horsepower"] = dftrain["horsepower"].fillna(0)
dftest["horsepower"] = dftest["horsepower"].fillna(0)

X_train = dftrain[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
y_train = dftrain["fuel_efficiency_mpg"]

X_test = dftest[["engine_displacement", "horsepower", "vehicle_weight", "model_year"]]
y_test = dftest["fuel_efficiency_mpg"]

model = Ridge(alpha=0.001)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", round(rmse, 3))

Test RMSE: 2.236
